Dataset is really regression oriented, too many continuous features

In [0]:
!pip install ucimlrepo

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pprint
import pyspark.sql.functions as F

In [0]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
appliances_energy_prediction = fetch_ucirepo(id=374) 
  
# data (as pandas dataframes) 
X = appliances_energy_prediction.data.features 
y = appliances_energy_prediction.data.targets 
  
# metadata 
pprint.pprint(appliances_energy_prediction.metadata) 
  
# variable information 
pprint.pprint(appliances_energy_prediction.variables) 


In [0]:
X.shape, y.shape

In [0]:
df = pd.concat([X, y], axis=1)
display(df)

In [0]:
df_spark = spark.createDataFrame(df)

In [0]:
# Null and NaN analysis
null_counts = df_spark.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_spark.columns])
display(null_counts)

In [0]:
# Data types and unique values
schema_info = [(c, t, df_spark.select(c).distinct().count()) for c, t in df_spark.dtypes]
display(pd.DataFrame(schema_info, columns=['column', 'type', 'unique_count']))

In [0]:
# Standard deviation for numeric columns
numeric_cols = [c for c, t in df_spark.dtypes if t in ['int', 'double', 'float', 'long']]
stddevs = df_spark.select([F.stddev(c).alias(c) for c in numeric_cols])
display(stddevs)

### Unilateral analysis
##### The Shapiro-Wilk test

- The Shapiro-Wilk test is a powerful statistical test to check if a dataset comes from a normal (bell-shaped) distribution, particularly good for smaller samples (n<5000). It works by comparing your sample data to a perfect normal distribution, producing a W statistic and a p-value, where a p-value > 0.05 suggests normality (no significant deviation), while a p-value <= 0.05 indicates the data is not normally distributed

In [0]:
import scipy.stats as stats

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
axes = axes.flatten()

for i, c in enumerate(numeric_cols[:16]):
    pdf = df_spark.select(c).dropna().toPandas()
    ax = axes[i]
    sns.histplot(pdf[c], kde=True, bins=30, ax=ax)
    ax.set_title(f'Normal Dist. of {c}')
    ax.set_xlabel(c)
    ax.set_ylabel('Frequency')
    # Normality test (Shapiro-Wilk)
    stat, p = stats.shapiro(pdf[c].sample(min(5000, len(pdf[c]))))
    ax.axhline(y=0.05, color='red', linestyle='--', label='p=0.05 threshold')
    ax.legend([f'p={p:.3f}'])
for j in range(i+1, 16):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [0]:
target_col = 'Appliances'
date_cols = ['date']

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_rainbow, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [0]:
display(df_spark.describe())

In [0]:
# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

TARGET_VAR = ['Appliances']
INITIAL_FEATURES = ['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4',
       'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9',
       'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility',
       'Tdewpoint', 'rv1', 'rv2']

In [0]:
# --- 2. Correlation Analysis and Initial Graphics ---
print("--- Step 2: Correlation Analysis (Pearson Coefficient & Graphics) ---")

correlation_matrix = df[INITIAL_FEATURES + TARGET_VAR].corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(17, 17))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Feature Correlation Matrix Heatmap')
plt.show()

In [0]:
date_col = ['date']
TARGET_VAR = ['Appliances']
INITIAL_FEATURES = ['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4',
       'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9',
       'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility',
       'Tdewpoint', 'rv1', 'rv2']

In [0]:
!pip install regex

In [0]:
import regex as re

In [0]:
df_spark_cleaned = df_spark.withColumn(
    "date",
    F.udf(lambda x: re.sub(r"^(.{10})(.*)$", r"\1 \2", x) if x else x, "string")("date")
)

In [0]:
display(df_spark_cleaned.head(3))

##### finding out of band data points

In [0]:
def plot_bollinger_out_of_band(df_spark, date_col, TARGET_VAR, INITIAL_FEATURES):
    numeric_feats = [feat for feat in INITIAL_FEATURES if feat in [c for c, t in df_spark.dtypes if t in ['int', 'double', 'float', 'long']]]
    features_to_plot = numeric_feats[:8]
    target = TARGET_VAR[0] if isinstance(TARGET_VAR, list) else TARGET_VAR

    out_of_band_dfs = []

    if date_col in df_spark.columns:
        for feat in features_to_plot:
            pdf = df_spark.select(date_col, feat).dropna().toPandas()
            pdf[date_col] = pd.to_datetime(pdf[date_col])
            pdf = pdf.sort_values(date_col)
            pdf['mean'] = pdf[feat].rolling(window=20).mean()
            pdf['std'] = pdf[feat].rolling(window=20).std()
            pdf['upper_band'] = pdf['mean'] + 2 * pdf['std']
            pdf['lower_band'] = pdf['mean'] - 2 * pdf['std']
            out_of_band = pdf[(pdf[feat] > pdf['upper_band']) | (pdf[feat] < pdf['lower_band'])].copy()
            out_of_band['feature'] = feat
            out_of_band_dfs.append(out_of_band[[date_col, feat, 'mean', 'std', 'upper_band', 'lower_band', 'feature']])

    return pd.concat(out_of_band_dfs, ignore_index=True) if out_of_band_dfs else pd.DataFrame()

In [0]:
out_of_band_df = plot_bollinger_out_of_band(df_spark_cleaned, 'date', TARGET_VAR, INITIAL_FEATURES)

In [0]:
onehot_df = out_of_band_df.pivot_table(index='date', columns='feature', aggfunc='size', fill_value=0)
onehot_df = (onehot_df > 0).astype(int).reset_index()
display(onehot_df)

In [0]:
onehot_df.rename(columns={col: f"{col}_ohe" for col in onehot_df.columns if col != 'date'}, inplace=True)
onehot_df

In [0]:
# no duplicates
display(df_spark_cleaned.groupBy('date').count().filter("count > 1"))

##### Flaging out of band

In [0]:
df_joined = df_spark_cleaned.join(
    spark.createDataFrame(onehot_df), 
    on='date', 
    how='left'
)
display(df_joined)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot count of out-of-band events by feature
plt.figure(figsize=(6,3))
sns.countplot(data=out_of_band_df, x='feature', order=out_of_band_df['feature'].value_counts().index)
plt.title('Out-of-Band Event Count by Feature')
plt.xlabel('Feature')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [0]:
# Plot out-of-band events over time for top 3 features
top_features = out_of_band_df['feature'].value_counts().index[:3]
for feat in top_features:
    plt.figure(figsize=(7,3))
    subset = out_of_band_df[out_of_band_df['feature'] == feat]
    sns.scatterplot(x='date', y=feat, data=subset, hue='upper_band', palette='coolwarm', legend=False)
    plt.title(f'Out-of-Band Values Over Time for {feat}')
    plt.xlabel('Date')
    plt.ylabel(feat)
    plt.xticks(rotation=45)
    plt.show()

In [0]:
print("df shape:", df.shape)
print("out_of_band_df shape:", out_of_band_df.shape)
print("onehot_df shape:",onehot_df.shape)
print("df_joined shape:",(df_joined.count(), len(df_joined.columns)))

##### Seasonality in Time Series

**Trend:** The trend shows a general direction of the time series data over a long period of time. A trend can be increasing(upward), decreasing(downward), or horizontal(stationary).

**Seasonality:** The seasonality component exhibits a trend that repeats with respect to timing, direction, and magnitude. Some examples include an increase in water consumption in summer due to hot weather conditions.

**Cyclical Component:** These are the trends with no set repetition over a particular period of time. A cycle refers to the period of ups and downs, booms and slums of a time series, mostly observed in business cycles. These cycles do not exhibit a seasonal variation but generally occur over a time period of 3 to 12 years depending on the nature of the time series.

**Irregular Variation:** These are the fluctuations in the time series data which become evident when trend and cyclical variations are removed. These variations are unpredictable, erratic, and may or may not be random.

**ETS Decomposition:** is used to separate different components of a time series. The term ETS stands for Error, Trend and Seasonality.

### Time Series terminology 



In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from scipy.optimize import curve_fit

**Dependence** It refers to the association of two observations of the same variable at prior time periods.

In [0]:
df_joined.columns

In [0]:
series = df_joined.select('date', 'T8').toPandas().set_index('date')['T8']


In [0]:
series_head_100 = series.head(100)
series_head_100

In [0]:
def script_dependence(series):
    print("--- 1. Dependence (ACF Plot) ---")
    # Plot the ACF to visualize dependencies at different lags
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_acf(series, lags=20, ax=ax)
    ax.set_title('Autocorrelation Function (ACF) Plot')
    plt.show()

script_dependence(series_head_100)



**Stationarity** It shows the mean value of the series that remains constant over the time period. If past effects accumulate and the values increase towards infinity then stationarity is not met.



In [0]:
def script_stationarity(series):
    print("\n--- 2. Stationarity (ADF Test) ---")
    # Perform the ADF test
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')

    # Conclusion based on p-value
    if result[1] <= 0.05:
        print("Conclusion: Reject the null hypothesis (H0). The series is likely stationary.")
    else:
        print("Conclusion: Fail to reject the null hypothesis (H0). The series is non-stationary.")

script_stationarity(series_head_100)


**Differencing** Differencing is used to make the series stationary and to control the auto-correlations. There may be some cases in time series analyses where we do not require differencing and over-differenced series can produce wrong estimates.



In [0]:
def script_differencing(series):
    print("\n--- 3. Differencing ---")
    # Apply first-order differencing
    differenced_series = series.diff().dropna()
    
    # Plot the differenced series
    plt.figure(figsize=(8, 4))
    plt.plot(differenced_series)
    plt.title('1st Order Differenced Series')
    plt.xlabel('Date')
    plt.ylabel('Differenced Value')
    plt.xticks(rotation=45)
    plt.show()

    # Re-run ADF test on differenced series
    print("ADF Test on Differenced Data:")
    script_stationarity(differenced_series)

script_differencing(series_head_100)


**Specification** It may involve the testing of the linear or non-linear relationships of dependent variables by using time series models such as ARIMA models.


In [0]:
def script_specification(series):
    print("\n--- 4. Specification (ACF/PACF Plots) ---")
    # Use the differenced series (assuming d=1) for AR (p) and MA (q) identification
    differenced_series = series.diff().dropna() 

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(differenced_series, lags=20, ax=axes[0], title='ACF of Differenced Series')
    plot_pacf(differenced_series, lags=20, ax=axes[1], title='PACF of Differenced Series', method="ywm")
    plt.show()
    print("Analyze the plots to determine appropriate p (PACF cutoff lag) and q (ACF cutoff lag) values for an ARIMA model.")

script_specification(series_head_100)



**Exponential Smoothing** Exponential smoothing in time series analysis predicts the one next period value based on the past and current value. It involves averaging of data such that the non-systematic components of each individual case or observation cancel out each other. The exponential smoothing method is used to predict the short term prediction.


In [0]:
def script_exponential_smoothing(series):
    print("\n--- 5. Exponential Smoothing ---")
    # Fit Simple Exponential Smoothing model
    # 'smoothing_level' (alpha) is the smoothing factor
    model = SimpleExpSmoothing(series).fit(smoothing_level=0.2, optimized=False)
    forecast = model.forecast(steps=5) # Forecast the next 5 periods

    print("Forecasted values for the next 5 periods:")
    print(forecast)
    
#     # Plotting the results
#     plt.figure(figsize=(8, 4))
#     plt.plot(series.index, series, label='Actual Data')
#     plt.plot(forecast.index, forecast, label='Exponential Smoothing Forecast', color='red', linestyle='--')
#     plt.title('Simple Exponential Smoothing Forecast')
#     plt.legend()
#     plt.xticks(rotation=45)
#     plt.show()

script_exponential_smoothing(series_head_100)



**Curve fitting** Curve fitting regression in time series analysis is used when data is in a non-linear relationship.



In [0]:
def script_curve_fitting(series):
    print("\n--- 6. Curve Fitting Regression ---")
    # Define a non-linear function (e.g., exponential growth)
    def exponential_func(x, a, b, c):
        return a * np.exp(b * x) + c

    x_data = np.arange(len(series))
    y_data = series.values

    # Fit the curve
    try:
        popt, pcov = curve_fit(exponential_func, x_data, y_data, p0=(1, 0.01, 1))
        
        # Plot the original data and the fitted curve
        plt.figure(figsize=(8, 4))
        plt.plot(series.index, y_data, 'b-', label='Actual Data')
        plt.plot(series.index, exponential_func(x_data, *popt), 'r--', label='Fitted Curve')
        plt.title('Curve Fitting (Exponential Regression)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.xticks(rotation=45)
        plt.show()
        print(f"Fitted parameters (a, b, c): {popt}")
    except RuntimeError as e:
        print(f"Could not fit the curve: {e}. The data might be too linear or complex for the chosen function.")

script_curve_fitting(series_head_100)


**ARIMA** stands for Auto Regressive Integrated Moving Average.

In [0]:
def script_curve_fitting(series):
    print("\n--- 6. Curve Fitting Regression ---")
    # Define a non-linear function (e.g., exponential growth)
    def exponential_func(x, a, b, c):
        return a * np.exp(b * x) + c

    x_data = np.arange(len(series))
    y_data = series.values

    # Fit the curve
    try:
        popt, pcov = curve_fit(exponential_func, x_data, y_data, p0=(1, 0.01, 1))
        
        # Plot the original data and the fitted curve
        plt.figure(figsize=(8, 4))
        plt.plot(series.index, y_data, 'b-', label='Actual Data')
        plt.plot(series.index, exponential_func(x_data, *popt), 'r--', label='Fitted Curve')
        plt.title('Curve Fitting (Exponential Regression)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.xticks(rotation=45)
        plt.show()
        print(f"Fitted parameters (a, b, c): {popt}")
    except RuntimeError as e:
        print(f"Could not fit the curve: {e}. The data might be too linear or complex for the chosen function.")

script_curve_fitting(series_head_100)


##### Trend and seasonality

In [0]:
df_small = df.sample(n=10, random_state=42)
display(df_small)

In [0]:
def plot_df(df, x, y, title="", xlabel='date', ylabel='T8', dpi=100):
    plt.figure(figsize=(15,4), dpi=dpi)
    plt.plot(x, y, color='blue')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()
    

plot_df(df_small, x=df_small['date'], y=df_small['T8'], title='Trend and Seasonality')

Number 8.

https://www.kaggle.com/code/prashant111/complete-guide-on-time-series-analysis-in-python